# Revue du dataset

Apercu rapide des 7 fichiers `data/*.parquet` generes par `generate_data.py` : structure, volumetrie et distributions cles de chacun.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = Path("data")

df_patient = pd.read_parquet(DATA_DIR / "patient.parquet")
df_sejour = pd.read_parquet(DATA_DIR / "sejour.parquet")
df_mouvement = pd.read_parquet(DATA_DIR / "mouvement.parquet")
df_document = pd.read_parquet(DATA_DIR / "document.parquet")
df_biologie = pd.read_parquet(DATA_DIR / "biologie.parquet")
df_medicament = pd.read_parquet(DATA_DIR / "medicament.parquet")
df_constante = pd.read_parquet(DATA_DIR / "constante.parquet")

fichiers = {
    "patient": df_patient,
    "sejour": df_sejour,
    "mouvement": df_mouvement,
    "document": df_document,
    "biologie": df_biologie,
    "medicament": df_medicament,
    "constante": df_constante,
}


def vc(series):
    """value_counts() sous forme de dict compact, sans le bruit np.int64(...)."""
    return series.value_counts().astype(int).to_dict()

## Vue d'ensemble

Un patient a un ou plusieurs sejours ; chaque sejour rassemble des mouvements, documents, resultats de biologie, administrations de medicaments et constantes vitales.

In [2]:
resume = pd.DataFrame({
    "fichier": list(fichiers.keys()),
    "n_lignes": [df.shape[0] for df in fichiers.values()],
    "n_colonnes": [df.shape[1] for df in fichiers.values()],
    "colonnes": [list(df.columns) for df in fichiers.values()],
})
resume

,fichier,n_lignes,n_colonnes,colonnes
0,patient,100,4,"[id_patient, age, sexe, date_deces]"
1,sejour,100,7,"[id_sejour, id_patient, date_entree, date_sort..."
2,mouvement,100,5,"[id_mouvement, id_sejour, uf, date_entree, dat..."
3,document,158,6,"[id_entrepot, id_patient, id_sejour, titre, ty..."
4,biologie,488,8,"[id_biologie, id_patient, id_sejour, uf, date_..."
5,medicament,329,11,"[id_medicament, id_patient, id_sejour, date_ad..."
6,constante,2684,7,"[id_constante, id_patient, id_sejour, type_con..."


## 1. `patient.parquet`

Un patient par cas source : age, sexe, date de deces eventuelle.

In [3]:
df_patient.head(3)

,id_patient,age,sexe,date_deces
0,PAT000001,62,M,NaT
1,PAT000002,57,F,NaT
2,PAT000003,37,F,NaT


In [4]:
print(f"{len(df_patient)} patients | sexe : {vc(df_patient['sexe'])} | deces : {df_patient['date_deces'].notna().mean():.0%}")
print(f"age : min {df_patient['age'].min()} - moyenne {df_patient['age'].mean():.0f} - max {df_patient['age'].max()}")

100 patients | sexe : {'M': 53, 'F': 47} | deces : 1%
age : min 5 - moyenne 55 - max 85


## 2. `sejour.parquet`

Un sejour hospitalier par patient : dates d'entree/sortie, UF, specialite.

In [5]:
df_sejour.head(3)

,id_sejour,id_patient,date_entree,date_sortie,uf_entree,uf_sortie,specialite
0,SEJ000001,PAT000001,2023-12-25 15:08:34,2023-12-31 15:08:34,1710,1710,MEDECINE INTERNE
1,SEJ000002,PAT000002,2023-08-13 12:53:17,2023-08-15 12:53:17,1130,1130,CANCERO ADULTE
2,SEJ000003,PAT000003,2024-09-04 22:15:21,2024-09-11 22:15:21,1850,1850,OBSTETRIQUE


In [6]:
duree = (df_sejour["date_sortie"] - df_sejour["date_entree"]).dt.days
print(f"{len(df_sejour)} sejours | duree (jours) : min {duree.min()} - moyenne {duree.mean():.1f} - max {duree.max()}")
print("Top 3 UF d'entree :", dict(list(vc(df_sejour["uf_entree"]).items())[:3]))

100 sejours | duree (jours) : min 1 - moyenne 8.8 - max 36
Top 3 UF d'entree : {'1130': 14, '1350': 10, '1940': 9}


## 3. `mouvement.parquet`

Mouvements intra-sejour (une UF par sejour dans cette version).

In [7]:
df_mouvement.head(3)

,id_mouvement,id_sejour,uf,date_entree,date_sortie
0,MVT000001,SEJ000001,1710,2023-12-25 15:08:34,2023-12-31 15:08:34
1,MVT000002,SEJ000002,1130,2023-08-13 12:53:17,2023-08-15 12:53:17
2,MVT000003,SEJ000003,1850,2024-09-04 22:15:21,2024-09-11 22:15:21


In [8]:
print(f"{len(df_mouvement)} mouvements, soit {df_mouvement.groupby('id_sejour').size().mean():.1f} par sejour en moyenne")

100 mouvements, soit 1.0 par sejour en moyenne


## 4. `document.parquet`

Documents cliniques (CRH/CRC/CRO/courrier/ordonnance) rattaches au sejour, au format HTML.

In [9]:
df_document.drop(columns=["texte_affichage"]).head(3)

,id_entrepot,id_patient,id_sejour,titre,type_document
0,DOC000001,PAT000001,SEJ000001,Compte-rendu d'hospitalisation - 2023-12-31,Compte-rendu d'hospitalisation
1,DOC000002,PAT000002,SEJ000002,Compte-rendu d'hospitalisation - 2023-08-15,Compte-rendu d'hospitalisation
2,DOC000003,PAT000003,SEJ000003,Compte-rendu d'hospitalisation - 2024-09-11,Compte-rendu d'hospitalisation


In [10]:
print(f"{len(df_document)} documents | types : {vc(df_document['type_document'])}")

158 documents | types : {"Compte-rendu d'hospitalisation": 100, 'Compte-rendu de consultation': 29, 'Compte-rendu operatoire': 29}


### Exemple de contenu d'un document

In [11]:
from IPython.display import HTML
HTML(df_document.iloc[0]["texte_affichage"])

## 5. `biologie.parquet`

Resultats de biologie rattaches au sejour : valeur numerique (avec unite) ou textuelle selon le type de resultat.

In [12]:
df_biologie.head(3)

,id_biologie,id_patient,id_sejour,uf,date_prelevement,valeur_numerique,valeur_texte,unite
0,BIO000001,PAT000001,SEJ000001,1710,2023-12-30 03:08:20,55.08,None,mmol/L
1,BIO000002,PAT000001,SEJ000001,1710,2023-12-26 06:04:10,135.37,None,%
2,BIO000003,PAT000001,SEJ000001,1710,2023-12-28 04:35:22,6.45,None,g/L


In [13]:
n_numerique = df_biologie["valeur_numerique"].notna().sum()
print(f"{len(df_biologie)} resultats | numeriques : {n_numerique} ({n_numerique / len(df_biologie):.0%}) | textuels : {len(df_biologie) - n_numerique}")
print("Unites (resultats numeriques) :", vc(df_biologie["unite"]))
print("Valeurs textuelles :", vc(df_biologie["valeur_texte"]))

488 resultats | numeriques : 367 (75%) | textuels : 121
Unites (resultats numeriques) : {'g/L': 61, '10^9/L': 59, 'UI/L': 54, 'umol/L': 54, '%': 53, 'mmol/L': 45, 'mg/L': 41}
Valeurs textuelles : {'normal': 30, 'positif': 24, 'negatif': 23, 'anormal': 23, 'non contributif': 21}


## 6. `medicament.parquet`

Administrations medicamenteuses rattachees au sejour : dose, voie d'administration, code UCD/ATC.

In [14]:
df_medicament.head(3)

,id_medicament,id_patient,id_sejour,date_administration,quantite_administree,unite,ucd,atc,voie_administration,conditionnelle,commentaire
0,MED000001,PAT000001,SEJ000001,2023-12-26 13:47:10,130,mg,3400921959219,B01AC06,intramusculaire,True,None
1,MED000002,PAT000001,SEJ000001,2023-12-30 18:34:05,16,mg,3400937246430,C09AA02,topique,False,traitement interrompu a la demande du patient
2,MED000003,PAT000001,SEJ000001,2023-12-26 09:19:18,58,mg,3400930096770,C10AA05,orale,False,None


In [15]:
print(f"{len(df_medicament)} administrations | voies : {vc(df_medicament['voie_administration'])}")
print(f"conditionnelles (si besoin) : {df_medicament['conditionnelle'].mean():.0%} | avec commentaire : {df_medicament['commentaire'].notna().sum()}")

329 administrations | voies : {'orale': 76, 'intramusculaire': 74, 'sous-cutanee': 62, 'topique': 59, 'intraveineuse': 58}
conditionnelles (si besoin) : 16% | avec commentaire : 41


## 7. `constante.parquet`

Constantes vitales mesurees pendant le sejour (poids, taille, temperature, frequence cardiaque, frequence respiratoire, saturation en oxygene, pression arterielle), au format long.

In [16]:
df_constante.head(3)

,id_constante,id_patient,id_sejour,type_constante,date,valeur,unite
0,CST000001,PAT000001,SEJ000001,poids,2023-12-31 00:05:45,101.0,kg
1,CST000002,PAT000001,SEJ000001,poids,2023-12-29 03:13:32,57.5,kg
2,CST000003,PAT000001,SEJ000001,poids,2023-12-26 01:16:20,48.0,kg


In [17]:
print(f"{len(df_constante)} mesures | types : {vc(df_constante['type_constante'])}")

2684 mesures | types : {'saturation_o2': 414, 'frequence_cardiaque': 411, 'pression_arterielle_diastolique': 408, 'pression_arterielle_systolique': 396, 'temperature': 377, 'frequence_respiratoire': 375, 'poids': 203, 'taille': 100}
